# **Class 2: ML potentials for molecular dynamics**

In this tutorial, we will use ``graph-pes``, a tool developed in our group for **training and working with graph-based machine learning interatomic potentials** (MLIPs). It provides a unified interface that allows us to explore a range of workflows, from training bespoke models to adapting large pre-trained foundation models.


Throughout this session, we will gain hands-on experience with key ideas and practical workflows in modern MLIP development, including:
1. **Training a bespoke model** from scratch
2. Using and **fine-tuning foundation models**
3. **Numerical validation** and model assessment
4. Using MLIPs to run **molecular dynamics** simulations

Example code and configuration files are provided throughout the notebook to guide you. The exercises are designed to encourage experimentation to explore model behaviour and understand the practical considerations involved in training and adapting MLIPs.

# **Getting started**

**1. Google Colab & runtime setup**

This tutorial is designed to run in Google Colab, so you don’t need to install anything locally.

Colab allows us to run notebooks on either CPU, GPU or TPU. For this tutorial, GPU/TPU acceleration is preferred for faster training.

❗ To enable GPU/TPU acceleration: click on Runtime menu ➡ change runtime type ➡ T4 GPU or v5e-1 TPU

⚠️ No GPU available?
In some cases you may not be assigned a GPU runtime, which is completely fine. All exercises can run on CPU and can still be completed in the allotted time, though training may take slightly longer.

**2. ```graph-pes``` setup**

We will be using ``graph-pes`` to handle model training and inference. ``graph-pes`` is a Python package that provides a simple interface for training and using graph neural network (GNN) interatomic potentials.


Useful links:

* [graph-pes documentation](https://vldgroup.github.io/graph-pes/)
* [GitHub repository](https://github.com/vldgroup/graph-pes)

``graph-pes`` supports training models from scratch (e.g. NequIP, PaiNN, MACE, TensorNet), custom architectures, architecture-agnostic validation pipelines, and the use of pre-trained foundation models (e.g. MACE, MatterSim, Orb) with minimal code changes.


In this tutorial, we will focus on practical usage: training, fine-tuning, and evaluating models rather than architectural details.

We first install ``graph-pes`` and its dependencies.

In [ ]:
!pip install graph-pes mace-torch

Check it's installed correctly by seeing if you have access to the ``graph-pes-train`` command.

In [ ]:
!graph-pes-train -h

# **Data preparation**

Throughout this tutorial, we will use a **carbon dataset** as a common reference for comparing different approaches, including models trained from scratch, zero-shot pretrained models, and fine-tuned models.


We begin by **loading the data** and then prepare it for modelling by creating **training, validation, and test sets**, as well as additional test subsets corresponding to different carbon phases.

We will use the **C-GAP-20U** dataset for carbon. This dataset was constructed in an iterative manner and contains around 6,000 structures spanning bulk crystalline and amorphous phases, crystal surfaces, and defective carbon configurations.


Further details can be found in the [original paper](https://doi.org/10.1063/5.0005084) and its Supplementary Information (https://doi.org/10.17863/CAM.84096).

In [ ]:
import ase.io
from load_atoms import load_dataset

structures = load_dataset("C-GAP-20U")
print(len(structures))

We can see the variety of structures in this dataset by inspecting their ``config_type``.

In [ ]:
from collections import Counter

config_counts = Counter(struct.info["config_type"] for struct in structures)

print("Number of structures per config type:")
for config_type, count in config_counts.items():
    print(f"- {config_type}: {count}")

Each structure in the dataset has an ``energy`` and ``forces`` label computed using **DFT with the optB88-vdW exchange–correlation functional**. This functional is well suited for carbon systems as it provides an accurate description of both covalent bonding and van der Waals interactions.

Here's a brief reminder of how we can extract these quantities from each structure:

In [ ]:
structures[0].info["energy"]

In [ ]:
structures[0].arrays["forces"].shape

❗ Note the shape of the ``forces`` label: each atom in the structure has an associated force vector containing three components corresponding to the Cartesian directions (x,y,z). As a result, the force array has shape (number_of_atoms, 3).

Similar to Class 1, we split the dataset into **training**, **validation**, and **test** sets. This allows us to separate the data used for learning from the data used for evaluation.

* **Training set**: used to optimise the model parameters during training.
* **Validation set**: used during training to monitor performance and guide decisions such as model selection or early stopping.
* **Test set**: held back entirely during training and used only for the final evaluation of model performance.

In [ ]:
train, val, test = structures.random_split([0.8, 0.1, 0.1], seed=42)

ase.io.write("train.xyz", train)
ase.io.write("valid.xyz", val)
ase.io.write("test.xyz", test)

In [ ]:
print(len(train), len(val), len(test))

You can preview these files through bash (as below), or through the Colab file browser on the left-hand side.

In [ ]:
!head train.xyz

# **Task 1: Training a bespoke model**

Now that we've saved our labelled structures to suitable files, we're ready to train a model.

We start by training a carbon model **from scratch**, meaning the model parameters are randomly initialised and learned only from the dataset, without using any prior pre-trained weights.

To train a ``graph-pes`` model we need a **configuration file** with all the model parameters specified.

We provide an example input file (in ``.yaml`` format), which specifies:

* the model architecture to instantiate and train.
* the training data (a random split of the C-GAP-20U dataset we just downloaded).
* the loss function, here a combination of a per-atom energy loss and a per-atom force loss.
* key training hyperparameters such as learning rate, batch size, and optimisation settings.



Download this config file using wget:

In [ ]:
%%bash

if [ ! -f train_bespoke.yaml ]; then
    wget https://raw.githubusercontent.com/nfragapane/Atomistic-ML-Tutorial/main/OxMLIP/train_bespoke.yaml -O train_bespoke.yaml
fi

Check out the structure of the config file, which will now be saved in the 'Files' section. Double click on it to open a pop-up window - you can customise it there.

## Training

The task here is to train your own bespoke model: **choose a model architecture and define its key hyperparameters**.

*  Choose an **architecture** (e.g. NequIP, MACE, TensorNet).
*  Consult the [documentation](https://vldgroup.github.io/graph-pes/) to understand which **hyperparameters** are relevant for your chosen architecture (e.g. ``cutoff``, ``radial features``, ``number of layers``), and specify them in the config file.
* You can also experiment with more **advanced training settings** such as the learning rate, batch size, or loss weighting.

Everyone should end up with a slightly different model. We will then compare results to see which configuration performs best in terms of accuracy. Make sure to keep your trained models safe, as you will need them again in a later tutorial.

⏲ We have limited ``max_epochs`` to 250 to reduce training time.

To train the model, we use the [graph-pes-train](https://vldgroup.github.io/graph-pes/cli/graph-pes-train/root.html) shell command.

In [ ]:
!graph-pes-train train_bespoke.yaml general/run_id=bespoke-train

## Model analysis
Once the model has finished training, we can load it and **evaluate its performance on the test set**.

You can see from the training logs that the best model (i.e. the set of weights that gave the lowest validation loss) has been saved as ``graph-pes-results/train-bespoke/model.pt``.

For more information on how to use ASE with ``graph-pes``, see the documentation here:


*   [ASE calculator](https://wiki.fysik.dtu.dk/ase/ase/calculators/calculators.html)
*   [graph-pes ASE interface](https://vldgroup.github.io/graph-pes/tools/ase.html)



In [ ]:
import torch
from graph_pes.models import load_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model = (
    load_model("graph-pes-results/bespoke-train/model.pt")  # load the model
    .to(device)  # move to GPU if available
    .eval()  # set to evaluation mode
)

# set up ASE calculator
calculator = best_model.ase_calculator()

We can now use the [GraphPESCalculator](https://vldgroup.github.io/graph-pes/tools/ase.html#graph_pes.utils.calculator.GraphPESCalculator) to act directly on [ase.Atoms](https://wiki.fysik.dtu.dk/ase/ase/atoms.html#module-ase.atoms) objects:

In [ ]:
# example of getting energies and forces using the calculator
test[0].calc = calculator
energy_pred = test[0].get_potential_energy()
forces_pred = test[0].get_forces()

We can see from a single data point that our model has done a reasonable job of learning the PES:

In [ ]:
calculator.get_potential_energy(test[0]), test[0].info["energy"]

``graph-pes`` provides a few utility functions for visualising model performance:

In [ ]:
import matplotlib.pyplot as plt
from graph_pes.utils.analysis import parity_plot

%config InlineBackend.figure_format = 'retina'

parity_plot(
    best_model,
    test,
    property="energy_per_atom",
    units="eV / atom",
    lw=0,
    s=12,
    color="crimson",
)
plt.xlim(-8, -2.5)
plt.ylim(-8, -2.5);

In [ ]:
parity_plot(
    best_model,
    test,
    property="forces",
    units="eV / Å",
    lw=0,
    s=2,
    alpha=0.5,
    color="crimson",
)

In [ ]:
from graph_pes.utils.analysis import dimer_curve

dimer_curve(best_model, system="CC", units="eV", rmin=0.85, rmax=4.0);

We’ll now compute the **RMSE for energies and forces** on the test set as a complete quantitative evaluation of model performance.

Once that’s done, we’ll compare results across the class to see who built the **best (and worst) performing models**, and start linking performance back to the architectural and training choices you made.

In [ ]:
import numpy as np

def compute_energy_force_rmse(structures, calculator):
    """
    Compute energy and force RMSE across test set.

    Energy RMSE: meV/atom
    Force RMSE: meV/Å
    """

    true_energies = []
    pred_energies = []

    true_forces = []
    pred_forces = []

    for atoms in structures:
        atoms.calc = calculator

        # predictions
        e_pred = atoms.get_potential_energy() / len(atoms)
        f_pred = atoms.get_forces()

        # references
        e_true = atoms.info["energy"] / len(atoms)
        f_true = atoms.get_array("forces")

        pred_energies.append(e_pred)
        true_energies.append(e_true)

        pred_forces.append(f_pred)
        true_forces.append(f_true)

    # convert to arrays
    true_energies = np.array(true_energies)
    pred_energies = np.array(pred_energies)

    true_forces = np.concatenate(true_forces).flatten()
    pred_forces = np.concatenate(pred_forces).flatten()

    # energy RMSE (meV/atom)
    energy_rmse = np.sqrt(
        np.mean((true_energies - pred_energies)**2)
    ) * 1000

    # force RMSE (meV/Å)
    force_rmse = np.sqrt(
        np.mean((true_forces - pred_forces)**2)
    ) * 1000

    return energy_rmse, force_rmse

In [ ]:
energy_rmse, force_rmse = compute_energy_force_rmse(test, calculator)
print(f"Energy RMSE: {energy_rmse:.2f} meV/atom")
print(f"Force RMSE: {force_rmse:.2f} meV/Å")

#**Task 2: Pre-trained foundation models**

In this section, we move from training models from scratch to using **pre-trained foundation models**.

In the context of atomistic machine learning, these are large models trained on broad and diverse datasets, designed to capture general chemical and structural knowledge that can then be reused or adapted to new systems.
Instead of learning everything from scratch, we can leverage these models directly, or fine-tune them on our own dataset to improve performance for a specific task.

We will focus in particular on the [MACE](https://doi.org/10.1063/5.0297006) family of foundation models, which are widely used and provide strong performance across a range of materials systems.

## Loading and using foundation models

Let's start by loading the `MACE-MP-0-small` model, using the [graph_pes.interfaces.mace_mp](https://vldgroup.github.io/graph-pes/interfaces/mace.html#graph_pes.interfaces.mace_mp) function, and use it to make predictions on some carbon structures.

In [ ]:
import torch
from graph_pes.interfaces import mace_mp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

mp0 = mace_mp("small").eval().to(device)
print("Number of parameters:", sum(p.numel() for p in mp0.parameters())) # gives an indication of model size

This `model` object is a [GraphPESModel](https://vldgroup.github.io/graph-pes/models/root.html) instance, and so can be used throughout the rest of the `graph-pes` ecosystem. For instance, we can use it as an ASE calculator to generate force predictions:

In [ ]:
from ase import Atoms
from graph_pes.graph_pes_model import GraphPESModel
from load_atoms import load_dataset
import matplotlib.pyplot as plt

def check_forces(model: GraphPESModel, structure: Atoms):
    calculator = model.ase_calculator()

    force_predictions = calculator.get_forces(structure)
    force_labels = structure.arrays["forces"]

    plt.figure(figsize=(2.5, 2.5))
    plt.scatter(force_labels.flatten(), force_predictions.flatten(), s=6)
    plt.axline((0, 0), slope=1, color="black", ls="--", lw=1)
    plt.gca().set_aspect("equal")
    plt.xlabel("True force / eV/Å")
    plt.ylabel("Predicted force / eV/Å");

check_forces(mp0, test[0])

In [ ]:
calculator = mp0.ase_calculator()
_, force_rmse = compute_energy_force_rmse(test, calculator)
print(f"Force RMSE: {force_rmse:.2f} meV/Å")

## Comparing foundation models

Using the unified interface provided through ``graph-pes``, you can easily swap between different variants of the MACE family of models and evaluate their performance on the same test set.

Your task is now to explore how **model size** and the **choice of pre-training data** affect model performance:
1. Load the ``small``, ``medium``, and ``large`` MACE models.
2. Compute and compare the force RMSE on the test set for each model using the same evaluation procedure as before.
3. Then repeat for MACE models trained on different pre-training datasets, for example:
* ``medium-mp-0``: MPTrj dataset
* ``medium-mpa-0``: MPTrj + sAlex dataset
* ``medium-omat-0``: OMat24 dataset

This will help you build intuition for how both model scale and pre-training data influence performance in practice.

Once you have your results, take a moment to compare the different models and think about:
* How does performance change with increasing model size?
* How might differences in pre-training datasets affect accuracy and transferability?
* What trade-offs exist between accuracy and computational cost?

We will discuss these observations as a group once everyone has completed the exercise.

In [ ]:
# write your code here...

#**Task 3: Fine-tuning**

From the previous section, we saw that the pre-trained foundation models can give reasonable force predictions, even on the C-GAP-20U dataset, despite being trained on data generated with different levels of DFT theory.

However, there are two important reasons why fine-tuning can still be necessary:

1. **Improving performance in a specific domain**:
Even though foundation models are trained on large and diverse datasets, their performance can vary depending on the target system or chemical space. This is closely related to what we observed when comparing different pre-training datasets — the choice of training data can significantly influence accuracy and transferability. Fine-tuning allows us to specialise a general model to a particular dataset, such as carbon structures in C-GAP-20U, and improve accuracy in that regime.

2. **Aligning different levels of theory (DFT mismatch)**:
A second, more subtle issue arises from differences in the underlying quantum mechanical reference data.
Different datasets may be generated using different exchange–correlation functionals, which define the reference energy scale. As a result, absolute energies are not directly transferable between datasets. For example, the energy of an isolated atom $\varepsilon_X$ depends on the chosen functional, and in general: $\varepsilon_X^{(a)} \neq \varepsilon_X^{(b)}$.

    In this tutorial, the C-GAP-20U dataset uses the **optB88-vdW** functional, which includes dispersion interactions, whereas the pre-trained MACE models were trained on datasets based on **PBE/PBE+U**.

    This mismatch leads to a systematic energy offset, which we can visualise using an energy parity plot below. While force predictions may still be accurate, the energy baseline can differ significantly.


In [ ]:
from graph_pes.utils.analysis import parity_plot

parity_plot(
    mp0,
    test,
    property="energy_per_atom",
    units="eV/atom",
    c="crimson",
    label="MP0"
)
plt.xlabel("Ground truth Energy (eV/atom)")
plt.ylabel("Predicted Energy (eV/atom)")
plt.legend(frameon=True);

## Training

To fine-tune with ``graph-pes``, we need a config file similar to the one we used for training from scratch.



In [ ]:
%%bash

if [ ! -f fine_tune.yaml ]; then
    wget https://raw.githubusercontent.com/nfragapane/Atomistic-ML-Tutorial/main/OxMLIP/fine_tune.yaml -O fine_tune.yaml
fi

In this task, you will **fine-tune a pre-trained foundation model on the C-GAP-20U dataset** to improve performance in this domain.
The task is to set up and run your own fine-tuning experiment:

* **Choose a foundation model** to fine-tune (e.g. ``small``, ``medium``, or ``large`` variants of the MACE models).
* Select a **subset of the training data**.
* **Adjust hyperparameters** such as learning rate, batch size, or loss weighting.

Everyone will end up fine-tuning a slightly different model. We will compare results to see how choices in foundation model selection, data subset size, and hyperparameters affect performance.

Make sure to keep your fine-tuned models safe, as you will need them again in a later tutorial.

⏲ As before, max_epochs is limited to keep training times manageable.

Hint:
1. Use a relatively small learning rate (e.g. of the order 10^-3/-4)
2. Keep your fine-tuning dataset relatively small. Successful fine-tuning can be done with 10s or 100s of structures.

In [ ]:
# Define your fine-tuning datasets
# Make sure there's no data leakage between train and val
ft_train, ft_valid = train[:50], train[50:60]
ft_train.write("ft_train.extxyz")
ft_valid.write("ft_valid.extxyz")

Start fine-tuning with the same ``graph-pes-train`` command as before:

In [ ]:
!graph-pes-train fine_tune.yaml

Load in fine-tuned model and evaluate on test set. Has your fine-tuning procedure corrected for the energy offset?

In [ ]:
from graph_pes.models import load_model

zeroshot= mace_mp("...").eval().to(device)  # change to be your corresponding zero-shot model

parity_plot(
    zeroshot,
    test,
    property="energy_per_atom",
    units="eV/atom",
)
plt.title("Zero-shot")
plt.show()

fine_tuned = load_model("graph-pes-results/fine-tune/model.pt").eval()
parity_plot(
    fine_tuned,
    test,
    property="energy_per_atom",
    units="eV/atom",
)
plt.title("Fine-tuned")

Calculate energy and force RMSEs of the zero-shot and fine-tuned models on the test set. Has your fine-tuning procedure improved the accuracy of the model?

In [ ]:
# get energy and force RMSEs...

Optional extra tasks:

*   **Fine-tuning vs training from scratch**: Compare your fine-tuned model with the bespoke model trained earlier. Which performs better for the same amount of data and compute?
*   **Investigate energy vs force improvements**: Does fine-tuning improve energies and forces equally? Which quantity changes most? How does the loss function weighting influence this?
*   **Domain-specific fine-tuning:** fine-tune using only crystalline or amorphous configurations, then compare performance on both domains before and after fine-tuning. Do you see any signs of catastrophic forgetting?
*   **How much data do you actually need?**: repeat fine-tuning with different training set sizes (e.g. 50, 100, 250, 500 structures). How does performance scale with dataset size? Is there a point of diminishing returns?
*   **Compare different starting foundation models**: fine-tune different pre-trained model families (e.g. MACE, MatterSim, Orb) using the same dataset and hyperparameters. Does the choice of starting model still matter after fine-tuning?
*   **Efficiency trade-offs**: compare training time and inference time across models. Is the most accurate model always the most practical choice?




#**Task 4: Molecular dynamics simulations**

So far, we have explored several approaches for building high-quality MLIPs and introduced basic numerical validation through energy and force errors. While these metrics are useful, the ultimate test of a model is often its behaviour in **downstream simulations**.

A model with low prediction error does not necessarily produce stable or physically meaningful dynamics. Molecular dynamics (MD) simulations provide a more demanding test by repeatedly using the model to predict forces as atoms evolve over time.


In this section, we will use the models trained earlier to run MD simulations and investigate how they behave in practice. We will use [ASE](https://vldgroup.github.io/graph-pes/tools/ase.html) as the MD engine, although ``graph-pes`` models can also interface with tools such as [LAMMPS](https://vldgroup.github.io/graph-pes/tools/lammps.html).


We will start with a few guided examples before exploring the behaviour of different models in simulation.

We will be using the ``ase.md`` module to run the MD simulations and ``ase.optimize``for the geometry optimisation. More information can be found at these links:


*   [ASE MD documentation](https://wiki.fysik.dtu.dk/ase/ase/md.html#module-ase.md)
*   [ASE Optimizers documentation](https://wiki.fysik.dtu.dk/ase/ase/optimize.html)

If you don't already have it installed, ``OVITO`` is a good tool for visualisation. You can install it by following the instructions on the [OVITO website](https://www.ovito.org/#download).

Before you start the simulations below, make sure you load your model in, set up the ASE calculator, and attach it to your desired starting configuration:

❗  You can use your trained-from-scratch model, your fine-tuned model, or a zero-shot model (which you load as above using the ``mace_mp`` module).

In [ ]:
from graph_pes.models import load_model
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = (
    load_model("graph-pes-results/fine-tune/model.pt")  # for example, load the fine-tuned model
    .to(device)  # move to GPU if available
    .eval()  # set to evaluation mode
)

# set up ASE calculator
calculator = model.ase_calculator()

# select your desired starting configuration
# we need to clean a copy of the Atoms object
initial_frame = structures[0].copy()
if "forces" in initial_frame.arrays:
    initial_frame.arrays.pop("forces")

# attach calculator to structure
initial_frame.calc = calculator

Test 1: test the stability of the MLIP by **annealing** (holding at constant temperature using the NVT ensemble) a selection of starting configurations at 300 K.

In [ ]:
import ase
from ase.md.velocitydistribution import (
    MaxwellBoltzmannDistribution,
    ZeroRotation,
    Stationary,
)
from ase.md import MDLogger
from ase.md.nvtberendsen import NVTBerendsen
from ase.io import read, write

Tinit = 300  # initial temperature in Kelvin

md_params = {
    "timestep": 1 * ase.units.fs,  # MD timestep in femtoseconds
    "taut": 100 * 0.5 * ase.units.fs,  # thermostat time constant
}
total_md_steps = 10000  # make sure change this to match the appropriate time scale of your system.

# initialise velocities
MaxwellBoltzmannDistribution(initial_frame, temperature_K=Tinit)
Stationary(initial_frame)
ZeroRotation(initial_frame)

# initialise dynamics object
dyn = NVTBerendsen(initial_frame, temperature_K=Tinit, **md_params)

# write trajectory function
def write_frame():
    dyn.atoms.write(
        f"trajectory.xyz", append=True
    )  # make sure the extension is xyz. Make sure to save the path of each trajectory file to avoid overwriting.


dyn.attach(write_frame, interval=100)  # set the frequency of writing to trajctory file

# set up the logger
dyn.attach(
    MDLogger(
        dyn,  # dynamics object
        initial_frame,  # intial configuration
        f"log.log",
        peratom=True,
        mode="a",
    ),
    interval=100,  # frequency of writing the log
)

# run the MD
dyn.run(total_md_steps)

Test 2: perform a **geometry optimisation**, this also is a good test for stability.

In [ ]:
from ase.optimize import LBFGS

opt = LBFGS(initial_frame, trajectory="opt.traj", logfile="opt.log")
opt.run(fmax=0.05, steps=100)  # run the optimiser until forces are below 0.05 eV/Ang, or 100 steps are reached.

# write the optimised structure to a file
write("optimised_structure.xyz", initial_frame)

*   Check the energy and forces of the optimised structures, how do they compare to the initial values?
*   Visualise the structures before and after optimisation- have they changed drastically?

Test 3: **graphitisation**, starting from an amorphous carbon configuration, you ramp the temperature from 300 K to a high temperature between 2000-4000 K and anneal for hundreds of ps. This is a less demanding simulation than the melt-quench (see below), but it is a good test to study subtle changes in the carbon structure that involve bond breaking and formation to form more ordered domains.

In [ ]:
# Adapt the annealing code for this purpose

# Make sure you change the T_init to be much higher

# And hold it for a longer time

Test 4: **melt-quench**, involving heating the system to a high temperature (> 5000 K), followed by rapid cooling to room temperature (e.g. 300 K). This is a common technique to study the melting and crystallisation of materials. It is considered a more demanding simulation than the graphitisation, as the large temperature ramp and rapid cooling can result in catastrophic failure if the model is not stable.

In [ ]:
import numpy as np
from ase import units
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary, ZeroRotation
from ase.md.nvtberendsen import NVTBerendsen
from ase.md import MDLogger
from ase.io import read, write
from ase.io.trajectory import Trajectory

# === Parameters ===
Tinit = 300      # Initial temperature (K)
Tmelt = 6000     # Melting temperature (K)
Tfin = 300       # Final (quench) temperature (K)

timestep = 1.0 * units.fs  # Time step
taut = 50.0 * units.fs     # Thermostat time constant
interval = 100             # Write output every 100 steps

# Simulation steps (adjust as needed)
heating_steps = 10000
melt_hold_steps = 50000
quench_steps = 20000

# === Load initial structure ===
MaxwellBoltzmannDistribution(initial_frame, temperature_K=Tinit)
Stationary(initial_frame)
ZeroRotation(initial_frame)

# === Initialize trajectory and logger ===
traj = "trajectory.xyz"
logfile = "md.log"

def write_frame():
    write(traj, dyn.atoms, append=True)

# === Create dynamics object ===
dyn = NVTBerendsen(initial_frame, temperature_K=Tinit, timestep=timestep, taut=taut)
dyn.attach(write_frame, interval=interval)
dyn.attach(MDLogger(dyn, initial_frame, logfile, mode="a", peratom=True), interval=interval)

# === Heating phase ===
print("Heating...")
heating_temps = np.linspace(Tinit, Tmelt, heating_steps // 10)
for T in heating_temps:
    dyn.set_temperature(T)
    dyn.run(10)

# === Melting / hold phase ===
print("Melting at 6000 K...")
dyn.set_temperature(Tmelt)
dyn.run(melt_hold_steps)

# === Quenching phase ===
print("Quenching...")
quench_temps = np.linspace(Tmelt, Tfin, quench_steps // 10)
for T in quench_temps:
    dyn.set_temperature(T)
    dyn.run(10)

print("Melt-quench complete.")

*   How does the structure change throughout the simulation? Hint: track the number of sp2 and sp3 carbons, the average bond length etc.

*   Check the final energy and forces of the structure after the simulation, how does this compare to the initial configuration?

*  Repeat these exercises for the different models we've looked at today – which are most stable?, do any give structures that are unreasonable?
